# ELMo 임베딩 확인 노트북

이 노트북은 `checkpoints/bilm/final_model.pt`를 로드해서 다음을 확인합니다.

- 문장 입력 후 토큰별 임베딩 추출
- 모델이 반환하는 레이어별 출력 shape/샘플 값 확인
- `bank`의 문맥별 임베딩 차이 확인 (river vs finance)

> 참고: 현재 체크포인트는 `SimpleLanguageModel` 형식이며 `lm_embeddings`를 2개 레이어 뷰로 반환합니다.
> (`[skip/base, top BiLSTM]`). 즉, 일반 ELMo 설명의 3개(`char-CNN + 2xBiLSTM`)가 그대로 노출되지는 않습니다.


In [ ]:
from pathlib import Path
import sys
import re
import torch
import torch.nn.functional as F

# 경로 설정
PROJECT_ROOT = Path.cwd().resolve()  # 이 노트북을 bilm-tf에서 열었다고 가정
if PROJECT_ROOT.name != 'bilm-tf':
    candidate = PROJECT_ROOT / 'bilm-tf'
    if candidate.is_dir():
        PROJECT_ROOT = candidate

CHECKPOINT_PATH = PROJECT_ROOT / 'checkpoints' / 'bilm' / 'final_model.pt'
VOCAB_PATH = PROJECT_ROOT / 'bilm' / 'data' / 'pretrain' / 'elmo' / 'vocab.txt'

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('PROJECT_ROOT =', PROJECT_ROOT)
print('CHECKPOINT_PATH exists =', CHECKPOINT_PATH.is_file())
print('VOCAB_PATH exists =', VOCAB_PATH.is_file())


In [ ]:
# 모델 로드
from bilm.src.simple_language_model import SimpleLanguageModel
from bilm.src.dataset.data import UnicodeCharsVocabulary

DEVICE = torch.device('cpu')
ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
options = ckpt['options']
state = ckpt['model_state_dict']

vocab_size = int(state['output_projection.weight'].shape[0])
model = SimpleLanguageModel(options, vocab_size)
model.load_state_dict(state, strict=True)
model.to(DEVICE)
model.eval()

max_chars = int(options['char_cnn']['max_characters_per_token'])
vocab = UnicodeCharsVocabulary(str(VOCAB_PATH), max_chars)

print('Loaded checkpoint:', CHECKPOINT_PATH)
print('char max length:', max_chars)
print('lstm n_layers(option):', options['lstm']['n_layers'])
print('lstm hidden dim(option):', options['lstm']['dim'])


In [ ]:
TOKEN_PATTERN = re.compile(r"\w+|[^\w\s]", flags=re.UNICODE)

def tokenize(text: str):
    return TOKEN_PATTERN.findall(text)

def encode_sentences_to_char_ids(sentences):
    tokenized = [tokenize(s) for s in sentences]
    encoded = [vocab.encode_chars(tokens, split=False) for tokens in tokenized]  # BOS/EOS 포함

    max_len = max(e.shape[0] for e in encoded)
    out = torch.zeros((len(encoded), max_len, max_chars), dtype=torch.long)
    for i, e in enumerate(encoded):
        out[i, :e.shape[0], :] = torch.from_numpy(e).long()

    return tokenized, out.to(DEVICE)

@torch.no_grad()
def run_model(sentences):
    tokenized, char_ids = encode_sentences_to_char_ids(sentences)
    out = model(char_ids)
    lm_embeddings = out['lm_embeddings']  # (B, L, T, D)
    mask = out['mask']
    return {
        'sentences': sentences,
        'tokens': tokenized,
        'char_ids': char_ids,
        'lm_embeddings': lm_embeddings,
        'mask': mask,
    }


In [ ]:
# 1) 문장 몇 개 넣어서 embedding 뽑기 + 레이어 출력 확인
sentences = [
    'I sat on the bank of the river and watched the ducks.',
    'She deposited her paycheck at the bank downtown.',
    'The movie was surprisingly touching and warm.'
]

res = run_model(sentences)
E = res['lm_embeddings']

print('lm_embeddings shape =', tuple(E.shape))
print('  - batch =', E.shape[0])
print('  - layers =', E.shape[1])
print('  - seq_len =', E.shape[2])
print('  - dim =', E.shape[3])

for li in range(E.shape[1]):
    layer = E[:, li]
    print(f'Layer {li}: shape={tuple(layer.shape)}, mean={layer.mean().item():.6f}, std={layer.std().item():.6f}')


In [ ]:
# 문장/토큰별로 특정 레이어 임베딩 벡터 접근 예시
sent_idx = 0
layer_idx = 1

tokens = res['tokens'][sent_idx]
# encode_chars는 BOS/EOS를 포함하므로 token index는 +1 오프셋
word = 'bank'
word_pos = [i for i, t in enumerate(tokens) if t.lower() == word]

print('Sentence:', res['sentences'][sent_idx])
print('Tokens  :', tokens)
print(f"'{word}' positions (without BOS):", word_pos)

if word_pos:
    p = word_pos[0] + 1
    vec = res['lm_embeddings'][sent_idx, layer_idx, p]
    print(f'Layer {layer_idx}, token={word}, vector shape =', tuple(vec.shape))
    print('first 10 values =', vec[:10])


In [ ]:
# 2) 핵심 포인트: bank 문맥(강둑 vs 금융) 임베딩 비교
s_river = 'I sat on the bank of the river and watched the ducks.'
s_fin = 'She deposited her paycheck at the bank downtown.'

pair = run_model([s_river, s_fin])
E = pair['lm_embeddings']

def get_token_vec(result, sent_idx, token_text, layer_idx):
    toks = result['tokens'][sent_idx]
    positions = [i for i, t in enumerate(toks) if t.lower() == token_text.lower()]
    if not positions:
        raise ValueError(f"token '{token_text}' not found in sentence[{sent_idx}]: {toks}")
    p = positions[0] + 1  # BOS offset
    return E[sent_idx, layer_idx, p], toks, positions[0]

for li in range(E.shape[1]):
    v1, t1, p1 = get_token_vec(pair, 0, 'bank', li)
    v2, t2, p2 = get_token_vec(pair, 1, 'bank', li)

    cos = F.cosine_similarity(v1.unsqueeze(0), v2.unsqueeze(0)).item()
    l2 = torch.norm(v1 - v2, p=2).item()

    print(f'[Layer {li}] cosine(bank_river, bank_finance) = {cos:.6f}, L2 distance = {l2:.6f}')


In [ ]:
# 3) 직접 문장 바꿔서 테스트
custom_sentences = [
    'The crane is flying over the wetland.',
    'The construction crane lifted steel beams.'
]

test = run_model(custom_sentences)
print('tokens[0]:', test['tokens'][0])
print('tokens[1]:', test['tokens'][1])
print('lm_embeddings shape:', tuple(test['lm_embeddings'].shape))

# 원하는 단어를 바꿔서 비교
target_word = 'crane'
for li in range(test['lm_embeddings'].shape[1]):
    toks0 = test['tokens'][0]
    toks1 = test['tokens'][1]
    p0 = [i for i, t in enumerate(toks0) if t.lower() == target_word][0] + 1
    p1 = [i for i, t in enumerate(toks1) if t.lower() == target_word][0] + 1
    v0 = test['lm_embeddings'][0, li, p0]
    v1 = test['lm_embeddings'][1, li, p1]
    cos = F.cosine_similarity(v0.unsqueeze(0), v1.unsqueeze(0)).item()
    print(f'Layer {li} cosine({target_word} sense1, sense2) = {cos:.6f}')
